# Data Preparation

## 1. Select data

Data source: https://www.kaggle.com/datasets/sturarods/anac-national-civil-aviation-agency-2000-2025?select=anac_brazil.csv

Selected attriblutes with rationale:
- `nr_passag_pagos`: revenue passengers - primary target, the core metric required.
- `nr_ano_mes_partida_real`: date in format YYYYMM for monthly aggregation.
- `ds_grupo_di`: filters for regular flights, excludes not regular (charters) and not productive (ferry/training).
- `ds_servico_tipo_linha`: filters for passenger flights, excludes cargo ones.
- `ds_natureza_tipo_linha`: separates domestic and international flights.
- `nm_pais_origem` and `nm_pais_destino`: restore missing values or not identified records in `ds_natureza_tipo_linha`.

The other attributes should be excluded because they are either derived from attributes mentioned above or do not carry useful information regarding national monthly passengers forecast.

## 2. Clean data

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
chunk_size = 100000 
chunks = []

reader = pd.read_csv(
    'anac_brazil.csv', 
    usecols=['nr_ano_mes_partida_real', 'nr_passag_pagos', 'ds_grupo_di', 'ds_servico_tipo_linha', 'ds_natureza_tipo_linha', 'nm_pais_origem', 'nm_pais_destino'], 
    chunksize=chunk_size
)

for chunk in reader:
    chunks.append(chunk)

df = pd.concat(chunks, ignore_index=True)

In [ ]:
chunk_size = 100000 
chunks = []


# filter regular flights
df_filtered = df[df['ds_grupo_di'] == 'REGULAR']

# identify domestic/international flights
not_identified = df_filtered['ds_natureza_tipo_linha'] == 'NÃO IDENTIFICADO'
international = (df_filtered['nm_pais_origem'] != 'BRASIL') | (df_filtered['nm_pais_destino'] != 'BRASIL')

df_filtered['ds_natureza_tipo_linha'] = np.select(
    [
        not_identified & international, 
        not_identified & ~international
    ], 
    [
        'INTERNACIONAL', 
        'DOMÉSTICA'
    ], 
    default=df_filtered['ds_natureza_tipo_linha']
)

# filter passenger flights
passenger_flight = (df_filtered['nr_passag_pagos'] > 0.0 & df_filtered['nr_passag_pagos'] <= 853.0) & (df_filtered['ds_servico_tipo_linha'].isin(['PASSAGEIRO', 'NÃO IDENTIFICADO']))
df_filtered = df_filtered[passenger_flight]

In [ ]:
df_filtered.head()

## 5. ...

The resulting train dataset will include:
- `year_month` (datetime) - month of actual departure (index).
- `flight_type` (object) - domestic/international flight.
- `revenue_passengers` (integer) - sum of `nr_passag_pagos` as total monthly passengers.